# BSRN station-to-archive format checker — complete notebook wrapper

This notebook runs the original BSRN `f_check` V3.4 logic from inside Jupyter for any station, month, and year file matching the BSRN convention `sssmmyy.dat`.

The notebook is station-independent. It does not hard-code `dra0125.dat`; any `.dat` file or folder of `.dat` files can be checked by changing `INPUTS` or `INPUT_DIR` below.

The original C checker remains the reference implementation. This reproduces the historical archive test exactly while making the workflow notebook-based, auditable, and batch-capable.

In [1]:
from pathlib import Path
import os
import re
import shutil
import subprocess
from dataclasses import dataclass, asdict
from typing import Iterable, Optional

try:
    import pandas as pd
except Exception:
    pd = None

## 1. Configuration

Set either `INPUTS` to one or more explicit `.dat` files, or set `INPUT_DIR` to scan a folder. The individual `.rep` files are copied to `OUTPUT_DIR` and, by default, the temporary `.rep` files created next to the input `.dat` files are removed. A combined report can also be written to `OUTPUT_DIR`.


In [2]:
INPUTS = []  # e.g. ["dra0125.dat"] oder leer lassen und input_dir angeben
INPUT_DIR = "bsrn_check_input"  # e.g. "bsrn_check_input"
OUTPUT_DIR = "bsrn_check_output"
FORCE_RECOMPILE = False

# If True, remove the temporary .rep file that the legacy C checker writes next to each input .dat file.
# The canonical reports will remain only in OUTPUT_DIR.
REMOVE_INPUT_SIDE_REPORTS = True

# If set to a filename, check_many() also writes one combined report in OUTPUT_DIR.
# Set to None to disable combined output.
COMBINED_REPORT_FILE = "combined_checks.rep"


## 2. Embedded original C checker

The cell below writes the original uploaded `f_check_V3_4.c` source to the notebook output folder. Keeping the source embedded makes this notebook portable and avoids relying on a separate local C file.

In [3]:
C_SOURCE = "/********************************************************************************/\n/***\tProgramname:\tF_CHECK.C\t\t\t\t\t      ***/\n/***\tAuthor:\t\tVlado Nespor\t\t\t\t\t      ***/\n/***\tVersion:\t3.3,  01-OCT-2000\t\t\t\t      ***/\n/***\t\t\t\t\t\t\t\t\t      ***/\n/***\tFunction:\tChecks station-to-archive data files for errors\t      ***/\n/***\t\t\tin format of metadata and atm. data and writes\t      ***/\n/***\t\t\ta report file\t\t\t\t\t      ***/\n/***\t\t\t\t\t\t\t\t\t      ***/\n/***\tInput:\t\tStation-to-archive filename1.dat [filename2.dat ...]  ***/\n/***\t\t\t\t\t\t\t\t\t      ***/\n/***\tOutput:\t\tfilename1.rep [filename2.rep ...]\t\t      ***/\n/***\t\t\t\t\t\t\t\t\t      ***/\n/***\tNote:\t\tPlease report problems, suggestions, etc. to:\t      ***/\n/***\t\t\t\tVlado Nespor\tnespor@geo.umnw.ethz.ch       ***/\n/***\t\t\tor\tGuido Mueller\tmuller@geo.umnw.ethz.ch\t      ***/\n/***\tCompilation:\tcc -o f_check f_check.c\t\t\t\t      ***/\n/********************************************************************************/\n/***\tModifications:\tv.3.0 -> 3.1:\t-log. records 310 and 330 changed to  ***/\n/***\t\t\t\t\t log. records 3010 and 3030           ***/\n/***\t\t\t\t\t-added log. record 3300               ***/\n/***\t                v.3.1 -> 3.2:\t-year check changed to 4 difits       ***/\n/***\t                v.3.2 -> 3.3:\t-handels 3-band and 4-band radiation  ***/\n/***\t\t\t\t\t instruments                          ***/\n/********************************************************************************/\n\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n\nvoid f_check_usage(char *name)\n{\nprintf (\"\\n Usage : %s filename1.dat [filename2.dat ...]\\n\\n\",name);\nprintf (\" Input : filename1.dat,filename2.dat,... - names of station-to-archive data\\n\");\nprintf (\"                                           files to be checked for format\\n\");\nprintf (\" Output: filename1.rep,filename2.rep,... - report files\\n\\n\");\nprintf (\" Note  :\tPlease report problems, suggestions, etc. to:\\n\");\nprintf (\"   \t\t\tVlado Nespor\tnespor@geo.umnw.ethz.ch \\n\");\nprintf (\"   \t\tor\tGuido Mueller\tmuller@geo.umnw.ethz.ch \\n\\n\");\nexit(1);\n}\n\nFILE *fin;\t\t\t/* pointer to inputfile \t      \t\t*/\nFILE *fout;\t\t\t/* pointer to outputfile\t      \t\t*/\n\n\nint main(int narg, char *arg[])\n\n{\nint OS = 3;\t/* Operating system: 1 = Windows, 2 = MacOS X, 3 = UNIX */\n\nchar file_out[100];\t\t/* name of report file\t\t\t\t*/\nint sta_id;\t\t\t/* station identification number\t\t*/\nint month;\t\t\t/* month in input file name\t\t\t*/\nint year;\t\t\t/* year in input file name\t\t\t*/\nint year4;\t\t\t/* 4 digit year in input file name\t\t*/\nint year_min=1992,year_max=2030;/* min. and max. legal years\t\t\t*/\nchar buffer[120];\t\t/* buffer for input line\t\t\t*/\nint rec_num;\t\t\t/* logical record number\t\t\t*/\nchar rec_name[20];\t\t/* logical record name\t\t\t\t*/\nchar msg1[1500];\t\t/* 1st field for error messages\t\t\t*/\nchar msg2[1500];\t\t/* 2nd field for error messages\t\t\t*/\nchar msg3[1500];\t\t/* 3rd field for error messages\t\t\t*/\nchar sformat[100];\t\t/* scan    format for record line\t\t*/\nchar pformat[100];\t\t/* print   format for record line\t\t*/\nchar xformat[150];\t\t/* correct format for record line\t\t*/\nint day0,day1;\t\t\t/* previous and current day of measurement\t*/\nint min0,min1;\t\t\t/* previous and current minute of measurement\t*/\nint seq0,seq1;\t\t\t/* previous and current seq. no. in logrec 1100\t*/\nint hei0,hei1;\t\t\t/* previous and current height in logrec 1100\t*/\nint band4=0;\t\t\t/* flag for 3-band or 4-band rad. instrument\t*/\n\nchar c_temp;\t\t\t/* temporary scan fields\t\t\t*/\nchar *pfilename;\t\t/* pointer to the filename (without whole path)\t*/\nint i1[30];\nfloat f1[30];\nchar c1[81],c2[81],c3[81],c4[81];\n\nint err=0;\t\t\t/* exit value from main: # of files with error\t*/\n\nint lines;\nint i,j;\nint count1;\nint count2;\nint count3;\nint buf_len;\nif(narg==1)\t\t\t/* if no arguments on command line\t\t*/\n\tf_check_usage(arg[0]);\n\nfin=fout=NULL;\t\t\t/* stream pointer initialization\t\t*/\n\n/********************************************************************************/\n/* main loop for all files specified at command line\t\t\t\t*/\n/********************************************************************************/\nfor(i=1;i<narg;i++)\t\t\n\t{\n\tif(fin!=NULL) fclose(fin);\t/* close files if opened\t\t*/\n\tif(fout!=NULL) fclose(fout);\n\n\tswitch(OS)\n\t{\n\t\tcase 1:\n\t\t\tpfilename=strrchr(arg[i],'\\\\');\n\t\t\tbreak;\n\t\tcase 2:\t\n\t\t\tpfilename=strrchr(arg[i],'/');\n\t\t\tbreak;\n\t\tcase 3:\t\n\t\t\tpfilename=strrchr(arg[i],'/');\n\t\t\tbreak;\n\t\tdefault:\t\n\t\t\tpfilename=strrchr(arg[i],'/');\n\t\t\tbreak;\n\t}\n\n\tif(pfilename==NULL)\n\t\tpfilename=arg[i];\n\telse\n\t\tpfilename++;\n\n\t/*======================================================================*/\n\t/* Check for legal input filenames\t\t\t\t\t*/\n\t/*======================================================================*/\n\tsprintf(buffer,\"%.2s\\0\",pfilename+3);\n\tmonth=atoi(buffer);\n\tif(month<1 || month>12)\n\t\t{\n\t\tstrcpy(msg1,\"*ERROR: Illegal month '%02d' - 4th and 5th characters\\n\");\n\t\tstrcat(msg1,\"        in input file name '%s'.\\n\\0\");\n\t\tprintf(msg1,month,pfilename);\n\t\terr=1;\n\t\tcontinue;\n\t\t}\n\n\tsprintf(buffer,\"%.2s\\0\",pfilename+5);\n\tyear=atoi(buffer);\n\t/* convert year to four digit number\t\t\t\t\t*/\n\tyear4=year+2000 - (int)((year+50)/100)*100;\n\tif(year4<year_min || year4>year_max)\n\t\t{\n\t\tstrcpy(msg1,\"*ERROR: Illegal year '%02d' - 6th and 7th characters\\n\");\n\t\tstrcat(msg1,\"        in input file name '%s'.\\n\");\n\t\tstrcat(msg1,\"        (Expected year should be between %4d and %4d.)\\n\\0\");\n\t\tprintf(msg1,year,pfilename,year_min,year_max);\n\t\terr=1;\n\t\tcontinue;\n\t\t}\n\n\tif(strncmp(pfilename+7,\".dat\\0\",5))\n\t\t{\n\t\tstrcpy(msg1,\"*ERROR: Illegal extension '%s' - from 8th character on\\n\");\n\t\tstrcat(msg1,\"        in input file name '%s'.\\n\");\n\t\tstrcat(msg1,\"        (Expected extension is '.dat'.)\\n\\0\");\n\t\tprintf(msg1,pfilename+7,pfilename);\n\t\terr=1;\n\t\tcontinue;\n\t\t}\n\n\t/************************************************************************/\n\t/* open input file and report file\t\t\t\t\t*/\n\t/************************************************************************/\n\tif((fin=fopen(arg[i],\"r\"))==NULL)\n\t\t{\n\t\tprintf(\"Error while opening '%s' file!\\n\",arg[i]);\n\t\terr=1;\n\t\tcontinue;\t\t/* continue with next file\t\t*/\n\t\t}\n\n\tstrcpy(file_out,arg[i]);\n\tsprintf(strrchr(file_out,'.'),\".rep\");\n\n\tif((fout=fopen(file_out,\"w\"))==NULL)\n\t\t{\n\t\tprintf(\"Error while opening '%s' file!\\n\",file_out);\n\t\terr=1;\n\t\tcontinue;\t\t/* continue with next file\t\t*/\n\t\t}\n\n\tstrcpy(msg1,\"\\n**********\\nFile name: %s\\n**********\\n\"); /* message\t*/\n\tprintf(msg1,pfilename);\t\t/* print message on screen\t\t*/\n\tfprintf(fout,msg1,pfilename);\t/* print message into report file\t*/\n\n\n\t/*======================================================================*/\n\t/* Check of the input file for line length (80 characters)\t\t*/\n\t/*======================================================================*/\n\tcount1=0;\n\tcount2=0;\n\tcount3=0;\n\n\tstrcpy(msg1,\"*ERROR: A line longer than 80 characters or missing line separator.\\n\\n\");\n\tstrcat(msg1,\"        For help the number and the first 20 characters of the line\\n\");\n\tstrcat(msg1,\"        causing the problem are printed below.\\n\");\n\tstrcat(msg1,\"        -------------------------------------------\\n\");\n\tstrcat(msg1,\"        | line number:  |  first 20 characters:   |\\n\");\n\tstrcat(msg1,\"        -------------------------------------------\\n\\0\");\n\n\twhile (fscanf(fin,\"%c\",&c_temp) != EOF)\n\t\t{\n\t\tcount1++;\t\t/* counter of characters in line\t*/\n\n\t\tif(count1<=20)\t\t/* store first 20 characters\t\t*/\n\t\t\tbuffer[count1-1]=c_temp;\n\n\t\tif(c_temp=='\\n')\t\t/* the end of line\t\t*/\n\t\t\t{\n\t\t\tcount2++;\t/* counter of lines in input file\t*/\n\t\t\tif(count1>81)\n\t\t\t\t{\n\t\t\t\tcount3++; /* counter of errors\t\t\t*/\n\t\t\t\tif(count3==1)\n\t\t\t\t\t{\n\t\t\t\t\tprintf(\"%s\",msg1);\n\t\t\t\t\tfprintf(fout,\"%s\",msg1);\n\t\t\t\t\t}\n\t\t\t\tstrcpy(msg2,\"        |   %6d      |  %.20s   |\\n\");\n\t\t\t\tstrcat(msg2,\"        -------------------------------------------\\n\");\n\t\t\t\tprintf(msg2,count2,buffer);\n\t\t\t\tfprintf(fout,msg2,count2,buffer);\n\n\t\t\t\tif(count3>4)\n\t\t\t\t\t{\n\t\t\t\t\tstrcpy(msg2,\"        NOTE: Only first 5 wrong lines are printed.\\n\");\n\t\t\t\t\tprintf(msg2);\n\t\t\t\t\tfprintf(fout,msg2);\n\t\t\t\t\tcount1=0;\n\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\tcount1=0;\n\t\t\t}\n\t\t}\n\tif(count1!=0)\n\t\t{\n\t\tstrcpy(msg2,\"*ERROR: Missing 'new line' at the end of file.\\n\");\n\t\tprintf(msg2);\n\t\tfprintf(fout,msg2);\n\t\terr=1;\n\t\tcontinue;\n\t\t}\n\tif(count3==0)\n\t\t{\n\t\tstrcpy(msg2,\"*Check for line length.......... OK\\n\");\n\t\tprintf(msg2);\n\t\tfprintf(fout,msg2);\n\t\t}\n\telse\n\t\t{\n\t\terr=1;\n\t\tcontinue;\n\t\t}\n\n\n\t/*======================================================================*/\n\t/* Check of the input file for illegal characters\t\t\t*/\n\t/*======================================================================*/\n\tcount1=0;\n\tcount2=0;\n\tcount3=0;\n\n\tstrcpy(msg1,\"*ERROR: An illegal character occured.\\n\\n\");\n\tstrcat(msg1,\"        Allowed ASCII characters:\\n\");\n\tstrcat(msg1,\"           - in logical records 1000 and less than 100:\\n\");\n\tstrcat(msg1,\"                         all printable characters from 'space' to '~',\\n\");\n\tstrcat(msg1,\"                         and in addition, for logical record 3 also \\n\");\n\tstrcat(msg1,\"                         'tabulator' (09 hex) is alloved.\\n\");\n\tstrcat(msg1,\"           - in all other logical records:\\n\");\n\tstrcat(msg1,\"                         'space','+','-','.' and digits from '0' to '9'.\\n\\n\");\n\tstrcat(msg1,\"        For help the logical record number, the number of the line from\\n\");\n\tstrcat(msg1,\"        beginning of the file, position of the illegal character in the\\n\");\n\tstrcat(msg1,\"        line, character itself (if printable) and its hexadecimal value\\n\");\n\tstrcat(msg1,\"        are printed below.\\n\");\n\tstrcat(msg1,\"        --------------------------------------------------------------------\\n\");\n\tstrcat(msg1,\"        |  log. record:  | line number:  |  position:  |  wrong character  |\\n\");\n\tstrcat(msg1,\"        --------------------------------------------------------------------\\n\\0\");\n\n\trec_num=0;\n\trewind(fin);\t\t/* go to the beginning of input file\t\t*/\n\twhile (fgets(buffer,100,fin) != NULL)\n\t\t{\n\t\tcount2++;\t\t/* counter of line in input file\t*/\n\t\tbuf_len=strlen(buffer);\n\n\t\tif (buffer[0]=='*')\t/* get logical record number\t\t*/\n\t\t\t{\n\t\t\tsprintf(rec_name,\"%.6s\\0\",buffer);\n\t\t\trec_num=atoi(rec_name+2);\n\t\t\tcontinue;\n\t\t\t}\n\n\t\tif(rec_num==0)\n\t\t\t{\n\t\t\tcount3++;\t/* counter of errors\t\t\t*/\n\t\t\tstrcpy(msg2,\"*ERROR: Illegal characters at the line %d of the file,\\n\");\n\t\t\tstrcat(msg2,\"        or wrong record number.\\n\\n\");\n\t\t\tprintf(msg2,count2-1);\n\t\t\tfprintf(fout,msg2,count2-1);\n\t\t\tbreak;\n\t\t\t}\n\n\t\t/****************************************************************/\n\t\t/* logical record dependent check\t\t\t\t*/\n\t\t/****************************************************************/\n\t\tif(rec_num>99 && rec_num!=1000)\n\t\t\t{\n\t\t\tfor(j=0;j<buf_len;j++)\n\t\t\t\tif(buffer[j]=='\\n' || buffer[j]=='-' || buffer[j]=='+' || buffer[j]=='.' || \n\t\t\t\t   buffer[j]==' ' || buffer[j]>='0' && buffer[j]<='9')\n\t\t\t\t\tcontinue;\n\t\t\t\telse\n\t\t\t\t\tbreak;\n\n\t\t\tif(j<buf_len) /* illegal character\t\t\t*/ \n\t\t\t\t{\n\t\t\t\t/* not printable character replace with ' '\t*/\n\t\t\t\tc_temp=(buffer[j]<' ' || buffer[j]>'~')?' ':buffer[j];\n\n\t\t\t\tcount3++;\t/* counter of errors\t\t*/\n\n\t\t\t\tif(count3==1)\n\t\t\t\t\t{\n\t\t\t\t\tprintf(msg1);\n\t\t\t\t\tfprintf(fout,msg1);\n\t\t\t\t\t}\n\t\t\t\tstrcpy(msg2,\"        |      %4d      |   %6d      |      %2d     |   %c   |  %02X (hex) |\\n\");\n\t\t\t\tstrcat(msg2,\"        --------------------------------------------------------------------\\n\");\n\t\t\t\tprintf(msg2,rec_num,count2,j+1,c_temp,buffer[j]);\n\t\t\t\tfprintf(fout,msg2,rec_num,count2,j+1,c_temp,buffer[j]);\n\n\t\t\t\tif(count3>4)\n\t\t\t\t\t{\n\t\t\t\t\tstrcpy(msg2,\"        NOTE: Only first 5 lines with illegal characters are printed.\\n\");\n\t\t\t\t\tprintf(msg2);\n\t\t\t\t\tfprintf(fout,msg2);\n\t\t\t\t\tcount1=0;\n\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t}\n\t\telse\n\t\t\t{\n\t\t\tfor(j=0;j<buf_len;j++)\n\t\t\t\tif(buffer[j]=='\\n' || buffer[j]>=' ' && buffer[j]<='~' || buffer[j]=='\\t' && rec_num==3)\n\t\t\t\t\tcontinue;\n\t\t\t\telse\n\t\t\t\t\tbreak;\n\t\t\tif(j<buf_len)\n\t\t\t\t{\n\t\t\t\tc_temp=' ';\n\t\t\t\tcount3++;\n\t\t\t\tif(count3==1)\n\t\t\t\t\t{\n\t\t\t\t\tprintf(\"%s\",msg1);\n\t\t\t\t\tfprintf(fout,\"%s\",msg1);\n\t\t\t\t\t}\n\t\t\t\tstrcpy(msg2,\"        |      %4d      |   %6d      |      %2d     |   %c   |  %02X (hex) |\\n\");\n\t\t\t\tstrcat(msg2,\"        --------------------------------------------------------------------\\n\");\n\t\t\t\tprintf(msg2,rec_num,count2,j+1,c_temp,buffer[j]);\n\t\t\t\tfprintf(fout,msg2,rec_num,count2,j+1,c_temp,buffer[j]);\n\n\t\t\t\tif(count3>4)\n\t\t\t\t\t{\n\t\t\t\t\tstrcpy(msg2,\"        NOTE: Only first 5 lines with illegal characters are printed.\\n\");\n\t\t\t\t\tprintf(msg2);\n\t\t\t\t\tfprintf(fout,msg2);\n\t\t\t\t\tcount1=0;\n\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\t}\n\t\t\t}\n\n\n\t\t}\n\tif(count3==0)\n\t\t{\n\t\tstrcpy(msg2,\"*Check for illegal characters... OK\\n\");\n\t\tprintf(msg2);\n\t\tfprintf(fout,msg2);\n\t\t}\n\telse\n\t\t{\n\t\terr=1;\n\t\tcontinue;\n\t\t}\n\n\n\t/*======================================================================*/\n\t/* Check of the input file for record line formats\t\t\t*/\n\t/*======================================================================*/\n\tcount1=0;\t\t\t/* indicator of line in logical record\t*/\n\tcount2=0;\t\t\t/* counter of line in input file\t*/\n\tcount3=0;\n\n\tstrcpy(msg1,\"*ERROR: Incorrect format.\\n\\n\");\n\tstrcat(msg1,\"        For each logical record the line number from the beginning of\\n\");\n\tstrcat(msg1,\"        the file and the message only for the first line with wrong\\n\");\n\tstrcat(msg1,\"        format is printed below.\\n\\n\\0\");\n\n\tstrcpy(msg2,\"     Log. record: %6d\\n\");\n\tstrcat(msg2,\"     Line number: %6d\\n\");\n\n\trec_num=0;\n\trewind(fin);\n\twhile (fgets(buffer,100,fin) != NULL)\n\t\t{\n\t\tcount2++;\t\t/* counter of line in input file\t*/\n\n\t\tif (buffer[0]=='*')\t/* get logical record number\t\t*/\n\t\t\t{\n\t\t\tsprintf(rec_name,\"%.6s\\0\",buffer);\n\t\t\trec_num=atoi(rec_name+2);\n\t\t\tday0=min0=count1=0; \t/* reset day, min. and counter\t*/\n\t\t\tseq0=hei0=0;\t\t/* reset seq. no. and height\t*/\n\t\t\tcontinue;\t\t/* of lines in record\t\t*/\n\t\t\t}\n\n\t\t/* do not check these logical records\t\t\t\t*/\n\t\tif(day0==-1 || rec_num==3 || rec_num==1000)\n\t\t\tcontinue;\n\n\t\tswitch(rec_num)\t/* switch according to logical record number\t*/\n\t\t\t{\n\t\t\tcase    1:\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d %4d %2d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ## #### ##|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&sta_id,&month,&year,i1);\n\t\t\t\t\t\tsprintf(file_out,pformat,sta_id,month,year,i1[0]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tdefault: count1++;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%d%d%d%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %9d %9d %9d %9d %9d %9d %9d %9d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ######### ######### ######### ######### ######### ######### ######### #########\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,i1+2,i1+3,i1+4,i1+5,i1+6,i1+7);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],i1[2],i1[3],i1[4],i1[5],i1[6],i1[7]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase    2:\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d %2d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ## ##|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,i1+2);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],i1[2]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=2;\n\t\t\t\t\t\tstrcpy(sformat,\"%38c%1c%20c%1c%20c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-38.38s %-20.20s %-20.20s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXX\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c2,&c_temp,c3);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1,c2,c3);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 2: count1=3;\n\t\t\t\t\t\tstrcpy(sformat,\"%15c%1c%50c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-15.15s %-50.50s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c2);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1,c2);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 3: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%80c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-80.80s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase    4:\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\tcase 6: if(count1==6) count1=7;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d %2d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ## ##|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,i1+2);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],i1[2]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=2;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ##|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 2: count1=3;\n\t\t\t\t\t\tstrcpy(sformat,\"%80c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-80.80s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 3: count1=4;\n\t\t\t\t\t\tstrcpy(sformat,\"%20c%1c%20c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-20.20s %-20.20s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXX|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c2);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1,c2);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 4: count1=5;\n\t\t\t\t\t\tstrcpy(sformat,\"%15c%1c%50c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-15.15s %-50.50s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c2);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1,c2);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 5: count1=6;\n\t\t\t\t\t\tstrcpy(sformat,\"%f%f%d%1c%5c\");\n\t\t\t\t\t\tstrcpy(pformat,\" %7.3f %7.3f %4d %-5.5s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ###.### ###.### #### XXXXX|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,f1,f1+1,i1,&c_temp,c1);\n\t\t\t\t\t\tsprintf(file_out,pformat,f1[0],f1[1],i1[0],c1);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 7: count1=7;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%d%d%d%d%d%d%d%d%d%d%d%d%d%d%d%d%d%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %3d %2d %3d %2d %3d %2d %3d %2d %3d %2d %3d %2d %3d %2d %3d %2d %3d %2d %3d %2d %3d %2d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ### ## ### ## ### ## ### ## ### ## ### ## ### ## ### ## ### ## ### ## ### ##|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,i1+2,i1+3,i1+4,i1+5,i1+6,i1+7,i1+8,i1+9,i1+10,\n\t\t\t\t\t\t\ti1+11,i1+12,i1+13,i1+14,i1+15,i1+16,i1+17,i1+18,i1+19,i1+20,i1+21);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],i1[2],i1[3],i1[4],i1[5],i1[6],i1[7],i1[8],i1[9],i1[10],\n\t\t\t\t\t\t\ti1[11],i1[12],i1[13],i1[14],i1[15],i1[16],i1[17],i1[18],i1[19],i1[20],i1[21]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase    5:\n\t\t\tcase    6:\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%1c%1c\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d %2d %c\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ## ## X|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,i1+2,&c_temp,c1);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],i1[2],c1[0]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=2;\n\t\t\t\t\t\tif(rec_num==5)\n\t\t\t\t\t\t\t{\n\t\t\t\t\t\t\tstrcpy(sformat,\"%30c%1c%25c%d%d%d%d%d%1c%5c\");\n\t\t\t\t\t\t\tstrcpy(pformat,\"%-30.30s %-25.25s %3d %2d %2d %2d %2d %-5.5s\\n\");\n\t\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXX ### ## ## ## ## XXXXX|\");\n\n\t\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c2,i1,i1+1,i1+2,i1+3,i1+4,&c_temp,c3);\n\t\t\t\t\t\t\tsprintf(file_out,pformat,c1,c2,i1[0],i1[1],i1[2],i1[3],i1[4],c3);\n\t\t\t\t\t\t\t}\n\t\t\t\t\t\telse\n\t\t\t\t\t\t\t{\n\t\t\t\t\t\t\tstrcpy(sformat,\"%30c%1c%25c%d%1c%5c\");\n\t\t\t\t\t\t\tstrcpy(pformat,\"%-30.30s %-25.25s %3d %-5.5s\\n\");\n\t\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXX ### XXXXX|\");\n\n\t\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c2,i1,&c_temp,c3);\n\t\t\t\t\t\t\tsprintf(file_out,pformat,c1,c2,i1[0],c3);\n\t\t\t\t\t\t\t}\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 2: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%80c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-80.80s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase    7:\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d %2d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ## ##|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,i1+2);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],i1[2]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 6: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%1c%1c%1c%1c%1c%1c%1c%1c%1c%1c%1c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%c %c %c %c %c %c\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"X X X X X X|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c1+1,&c_temp,c1+2,&c_temp,c1+3,&c_temp,c1+4,&c_temp,c1+5);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1[0],c1[1],c1[2],c1[3],c1[4],c1[5]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tdefault: count1++;\n\t\t\t\t\t\tstrcpy(sformat,\"%80c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-80.80s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase    8:\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%1c%1c\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d %2d %c\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ## ## X|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,i1+2,&c_temp,c1);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],i1[2],c1[0]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=2;\n\t\t\t\t\t\tstrcpy(sformat,\"%30c%1c%15c%1c%18c%1c%8c%d\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-30.30s %-15.15s %-18.18s %-8.8s %5d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXX XXXXXXXX #####\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c2,&c_temp,c3,&c_temp,c4,i1);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1,c2,c3,c4,i1[0]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 2: count1=3;\n\t\t\t\t\t\tstrcpy(sformat,\"%80c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-80.80s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 3: count1=4;\n\t\t\t\t\t\tif(sscanf(buffer,\"%s%s%s%s%s%s%s%s%s%s%s%s\",c1,c1,c1,c1,c1,c1,c1,c1,c1,c1,c1,c1)<=10)\n\t\t\t\t\t\t\t{\n\t\t\t\t\t\t\tband4=0; /* 3-band rad. instrument */\n\t\t\t\t\t\t\tstrcpy(sformat,\"%d%d%f%f%f%f%f%f%d%d\");\n\t\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d %7.3f %7.3f %7.3f %7.3f %7.3f %7.3f %2d %2d\\n\");\n\t\t\t\t\t\t\tstrcpy(xformat,\" ## ## ###.### ###.### ###.### ###.### ###.### ###.### ## ##|\");\n\n\t\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,f1,f1+1,f1+2,f1+3,f1+4,f1+5,i1+2,i1+3);\n\t\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],f1[0],f1[1],f1[2],f1[3],f1[4],f1[5],i1[2],i1[3]);\n\t\t\t\t\t\t\t}\n\t\t\t\t\t\telse\n\t\t\t\t\t\t\t{\n\t\t\t\t\t\t\tband4=1; /* 4-band rad. instrument */\n\t\t\t\t\t\t\tstrcpy(sformat,\"%d%d%f%f%f%f%f%f%f%f%d%d\");\n\t\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d %7.3f %7.3f %7.3f %7.3f %7.3f %7.3f %7.3f %7.3f %2d %2d\\n\");\n\t\t\t\t\t\t\tstrcpy(xformat,\" ## ## ###.### ###.### ###.### ###.### ###.### ###.### ###.### ###.### ## ##|\");\n\n\t\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,f1,f1+1,f1+2,f1+3,f1+4,f1+5,f1+6,f1+7,i1+2,i1+3);\n\t\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],f1[0],f1[1],f1[2],f1[3],f1[4],f1[5],f1[6],f1[7],i1[2],i1[3]);\n\t\t\t\t\t\t\t}\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 4: count1=5;\n\t\t\t\t\t\tstrcpy(sformat,\"%30c%1c%40c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-30.30s %-40.40s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXX XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c2);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1,c2);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 5:\n\t\t\t\t\tcase 6:\n\t\t\t\t\tcase 7: count1++;\n\t\t\t\t\t\tstrcpy(sformat,\"%8c%1c%8c%d%f%f\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-8.8s %-8.8s %2d %12.4f %12.4f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXX XXXXXXXX ## #######.#### #######.####|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1,&c_temp,c2,i1,f1,f1+1);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1,c2,i1[0],f1[0],f1[1]);\n\n\t\t\t\t\t\t/* 4-band rad. instrument: read one line more with the same format */\n\t\t\t\t\t\tif(band4==1 && count1==8)\n\t\t\t\t\t\t\t{\n\t\t\t\t\t\t\tcount1--;\n\t\t\t\t\t\t\tband4=0;\n\t\t\t\t\t\t\t}\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 8:\n\t\t\t\t\tcase 9: count1++;\n\t\t\t\t\t\tif(count1==10) count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%80c\");\n\t\t\t\t\t\tstrcpy(pformat,\"%-80.80s\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,c1);\n\t\t\t\t\t\tsprintf(file_out,pformat,c1);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase    9:\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%d%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %2d %2d %9d %5d %2d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ## ## ######### ##### ##|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,i1+1,i1+2,i1+3,i1+4,i1+5);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],i1[1],i1[2],i1[3],i1[4],i1[5]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase  100:\n\t\t\t\tlines=0;\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%f%d%d%d%f%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -### -##.# -### -###   -### -##.# -### -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%f%d%d%d%f%d%d%f%f%d\");\n\t\t\t\t\t\tstrcpy(pformat,\"           %4d %5.1f %4d %4d   %4d %5.1f %4d %4d    %5.1f %5.1f %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"           -### -##.# -### -###   -### -##.# -### -###    -##.# -##.# -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5,f1+2,f1+3,i1+6);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5],f1[2],f1[3],i1[6]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase  200:\n\t\t\tcase  300:\n\t\t\t\tlines=1;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%f%d%d%d%f%d%d%d%f%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -### -##.# -### -###   -### -##.# -### -###   -### -##.# -### -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5,i1+6,f1+2,i1+7,i1+8);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5],i1[6],f1[2],i1[7],i1[8]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase  400:\n\t\t\t\tlines=0;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%f%d%d%d%f%d%d%d%f%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -### -##.# -### -###   -### -##.# -### -###   -### -##.# -### -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5,i1+6,f1+2,i1+7,i1+8);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5],i1[6],f1[2],i1[7],i1[8]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=2;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%f%d%d%d%f%d%d%d%f%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\"           %4d %5.1f %4d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"           -### -##.# -### -###   -### -##.# -### -###   -### -##.# -### -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5,i1+6,f1+2,i1+7,i1+8);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5],i1[6],f1[2],i1[7],i1[8]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 2: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%f%d%d%d%f%d%d%d%f%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\"           %4d %5.1f %4d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"           -### -##.# -### -###   -### -##.# -### -###   -### -##.# -### -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5,i1+6,f1+2,i1+7,i1+8);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5],i1[6],f1[2],i1[7],i1[8]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase  500:\n\t\t\t\tlines=0;\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%f%f%f%f%f%f%f%f\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## #### -##.# -##.# -##.# -##.# -##.# -##.# -##.# -##.#|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,f1,f1+1,f1+2,f1+3,f1+4,f1+5,f1+6,f1+7);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,f1[0],f1[1],f1[2],f1[3],f1[4],f1[5],f1[6],f1[7]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%f%f%f%f%f%f%f%f%f%f%f%f\");\n\t\t\t\t\t\tstrcpy(pformat,\"         %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f %5.1f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"         -##.# -##.# -##.# -##.# -##.# -##.# -##.# -##.# -##.# -##.# -##.# -##.#|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,f1,f1+1,f1+2,f1+3,f1+4,f1+5,f1+6,f1+7,f1+8,f1+9,f1+10,f1+11);\n\t\t\t\t\t\tsprintf(file_out,pformat,f1[0],f1[1],f1[2],f1[3],f1[4],f1[5],f1[6],f1[7],f1[8],f1[9],f1[10],f1[11]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase 1100:\n\t\t\t\tlines=1;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%d%d%f%f%d%d%f\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %4d %4d %5d %5.1f %6.1f %3d %3d %4.1f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -### -### -#### -##.# -###.# -## -## -#.#|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,&seq1,i1,&hei1,f1,f1+1,i1+1,i1+2,f1+2);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,seq1,i1[0],hei1,f1[0],f1[1],i1[1],i1[2],f1[2]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase 1200:\n\t\t\t\tlines=1;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,i1);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,i1[0]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase 1300:\n\t\t\t\tlines=1;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%d%f%f%f%f\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %2d %5d %5.1f   %6.3f %6.3f %6.3f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -# ##### -##.#   -#.### -#.### -#.###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,i1,i1+1,f1,f1+1,f1+2,f1+3);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,i1[0],i1[1],f1[0],f1[1],f1[2],f1[3]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase 1500:\n\t\t\t\tlines=1;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%d%d%d%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %4d %4d %4d   %4d %4d %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -### -### -###   -### -### -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,i1,i1+1,i1+2,i1+3,i1+4,i1+5);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,i1[0],i1[1],i1[2],i1[3],i1[4],i1[5]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tcase 3010:\n\t\t\t\tlines=0;\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%f%d%d%d%f%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -### -##.# -### -###   -### -##.# -### -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%f%d%d%d%f%d%d%f%f%d\");\n\t\t\t\t\t\tstrcpy(pformat,\"           %4d %5.1f %4d %4d   %4d %5.1f %4d %4d    %5.1f %5.1f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"           -### -##.# -### -###   -### -##.# -### -###    -##.# -##.#|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5,f1+2,f1+3);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5],f1[2],f1[3]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\t\t\tcase 3025:\n\t\t\tcase 3030:\n\t\t\t\tlines=0;\n\t\t\t\tswitch(count1) \t/* switch according to line \t*/\n\t\t\t\t\t{\t/* in record\t\t\t*/\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%f%d%d%d%f%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -### -##.# -### -###   -### -##.# -### -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%f%d%d%d%f%d%d%f%f%d\");\n\t\t\t\t\t\tstrcpy(pformat,\"           %4d %5.1f %4d %4d   %4d %5.1f %4d %4d    %5.1f %5.1f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"           -### -##.# -### -###   -### -##.# -### -###    -##.# -##.#|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5,f1+2,f1+3);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5],f1[2],f1[3]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\t\t\tcase 3300:\t/* NEW RECORD COMES HERE */\n\t\t\t\tlines=0;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=1;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%d%f%d%d%d%f%d%d\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d   %4d %5.1f %4d %4d   %4d %5.1f %4d %4d\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## ####   -### -##.# -### -###   -### -##.# -### -###|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5]);\n\n\t\t\t\t\t\tbreak;\n\n\t\t\t\t\tcase 1: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%f%d%d%d%f%d%d%f%f\");\n\t\t\t\t\t\tstrcpy(pformat,\"           %4d %5.1f %4d %4d   %4d %5.1f %4d %4d    %5.1f %5.1f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\"           -### -##.# -### -###   -### -##.# -### -###    -##.# -##.#|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,i1,f1,i1+1,i1+2,i1+3,f1+1,i1+4,i1+5,f1+2,f1+3);\n\t\t\t\t\t\tsprintf(file_out,pformat,i1[0],f1[0],i1[1],i1[2],i1[3],f1[1],i1[4],i1[5],f1[2],f1[3]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\t\t\tcase 4000:\n\t\t\t\tlines=1;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%f%f%f%f%f%f%f%f%f%f\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d %6.2f %6.2f %6.2f %6.2f %6.1f  %6.2f %6.2f %6.2f %6.2f %6.1f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## #### -##.## -##.## -##.## -##.## -###.#  -##.## -##.## -##.## -##.## -###.#|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,f1,f1+1,f1+2,f1+3,f1+4,f1+5,f1+6,f1+7,f1+8,f1+9,f1+10);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,f1[0],f1[1],f1[2],f1[3],f1[4],f1[5],f1[6],f1[7],f1[8],f1[9],f1[10]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\t\t\tcase 4010:\n\t\t\t\tlines=1;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%f%f%f%f%f%f%f%f%f%f\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d %6.2f %6.2f %6.2f %6.2f %6.1f  %6.2f %6.2f %6.2f %6.2f %6.1f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## #### -##.## -##.## -##.## -##.## -###.#  -##.## -##.## -##.## -##.## -###.#|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,f1,f1+1,f1+2,f1+3,f1+4,f1+5,f1+6,f1+7,f1+8,f1+9,f1+10);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,f1[0],f1[1],f1[2],f1[3],f1[4],f1[5],f1[6],f1[7],f1[8],f1[9],f1[10]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\t\t\tcase 4030:\n\t\t\t\tlines=1;\n\t\t\t\tswitch(count1)\n\t\t\t\t\t{\n\t\t\t\t\tcase 0: count1=0;\n\t\t\t\t\t\tstrcpy(sformat,\"%d%d%f%f%f%f%f%f%f%f%f%f\");\n\t\t\t\t\t\tstrcpy(pformat,\" %2d %4d %6.2f %6.2f %6.2f %6.2f %6.1f  %6.2f %6.2f %6.2f %6.2f %6.1f\\n\");\n\t\t\t\t\t\tstrcpy(xformat,\" ## #### -##.## -##.## -##.## -##.## -###.#  -##.## -##.## -##.## -##.## -###.#|\");\n\n\t\t\t\t\t\t/* scan numbers and print them with correct format */\n\t\t\t\t\t\tsscanf(buffer,sformat,&day1,&min1,f1,f1+1,f1+2,f1+3,f1+4,f1+5,f1+6,f1+7,f1+8,f1+9,f1+10);\n\t\t\t\t\t\tsprintf(file_out,pformat,day1,min1,f1[0],f1[1],f1[2],f1[3],f1[4],f1[5],f1[6],f1[7],f1[8],f1[9],f1[10]);\n\n\t\t\t\t\t\tbreak;\n\t\t\t\t\t}\n\t\t\t\tbreak;\n\n\t\t\tdefault:\n\t\t\t\tcount3++;\n\t\t\t\tstrcpy(msg3,\"*ERROR: Incorrect record number.\\n\\n\");\n\t\t\t\tprintf(msg3);\n\t\t\t\tfprintf(fout,msg3);\n\t\t\t\tprintf(msg2,rec_num,count2-1);\n\t\t\t\tfprintf(fout,msg2,rec_num,count2-1);\n\t\t\t\tday0=-1;\t/* indicator that there was\t*/\n\t\t\t\t\t\t/* already error in this logrec\t*/\n\t\t\t\tcontinue;\n\t\t\t}\n\n\t\t/* empty line\t\t\t\t\t\t\t*/\n\t\tif(buffer[0]=='\\n')\n\t\t\t{\n\t\t\tcount3++;\t/* counter of errors\t\t\t*/\n\n\t\t\tif(count3==1)\n\t\t\t\t{\n\t\t\t\tprintf(\"%s\",msg1);\n\t\t\t\tfprintf(fout,\"%s\",msg1);\n\t\t\t\t}\n\t\t\tprintf(msg2,rec_num,count2);\n\t\t\tfprintf(fout,msg2,rec_num,count2);\n\t\t\tstrcpy(msg3,\"        +Empty line...\\n\\n\");\n\t\t\tprintf(msg3);\n\t\t\tfprintf(fout,msg3);\n\t\t\tday0=-1;\t\t/* indicator that there was\t*/\n\t\t\t\t\t\t/* already error in this logrec\t*/\n\t\t\tcontinue;\n\t\t\t}\n\n\t\t/* wrong seq. no. or height of measurement in logrec 1100\t*/\n\t\tif(rec_num==1100)\n\t\t\t{\n\t\t\tif(count1+lines==1 && (seq0>=seq1 || hei0>=hei1) && day0==day1 && min0==min1)\n\t\t\t\t{\n\t\t\t\tcount3++;\n\t\t\t\tif(count3==1)\n\t\t\t\t\t{\n\t\t\t\t\tprintf(\"%s\",msg1);\n\t\t\t\t\tfprintf(fout,\"%s\",msg1);\n\t\t\t\t\t}\n\t\t\t\tprintf(msg2,rec_num,count2);\n\t\t\t\tfprintf(fout,msg2,rec_num,count2);\n\t\t\t\tstrcpy(msg3,\"        +Wrong seq. no. or height... (both have to be increasing)\\n\");\n\t\t\t\tprintf(msg3);\n\t\t\t\tfprintf(fout,msg3);\n\t\t\t\tstrcpy(msg3,\"        ------------------------------------------------------------\\n\");\n\t\t\t\tstrcat(msg3,\"        |            |    day     |  minute  | seq. no. |  height  |\\n\");\n\t\t\t\tstrcat(msg3,\"        ------------------------------------------------------------\\n\");\n\t\t\t\tprintf(msg3);\n\t\t\t\tfprintf(fout,msg3);\n\t\t\t\tstrcpy(msg3,\"        | Previous   |  %5d     | %6d   | %6d   | %6d   |\\n\");\n\t\t\t\tprintf(msg3,day0,min0,seq0,hei0);\n\t\t\t\tfprintf(fout,msg3,day0,min0,seq0,hei0);\n\t\t\t\tstrcpy(msg3,\"        | Current    |  %5d     | %6d   | %6d   | %6d   |\\n\");\n\t\t\t\tstrcat(msg3,\"        ------------------------------------------------------------\\n\\n\");\n\t\t\t\tprintf(msg3,day1,min1,seq1,hei1);\n\t\t\t\tfprintf(fout,msg3,day1,min1,seq1,hei1);\n\t\t\t\tday0=-1;\n\t\t\t\tcontinue;\n\t\t\t\t}\n\t\t\telse\n\t\t\t\t{\n\t\t\t\tseq0=seq1;/* set cur. seq. no and heig. to prev.*/\n\t\t\t\thei0=hei1;\n\t\t\t\t}\n\t\t\t}\n\n\t\t/* wrong day and minute of measurement\t\t\t\t*/\n\t\tif(rec_num>=100)\n\t\t\t{\n\t\t\tif(count1+lines==1 && (day0>day1 || day0==day1 && (min0>=min1 && rec_num!=1100 ||\n\t\t                                                                   min0> min1 && rec_num==1100)))\n\t\t\t\t{\n\t\t\t\tcount3++;\n\t\t\t\tif(count3==1)\n\t\t\t\t\t{\n\t\t\t\t\tprintf(\"%s\",msg1);\n\t\t\t\t\tfprintf(fout,\"%s\",msg1);\n\t\t\t\t\t}\n\t\t\t\tprintf(msg2,rec_num,count2);\n\t\t\t\tfprintf(fout,msg2,rec_num,count2);\n\t\t\t\tstrcpy(msg3,\"        +Wrong day and minute... \");\n\t\t\t\tif(rec_num!=1100)\n\t\t\t\t\tstrcat(msg3,\"(time of measurement have to be increasing)\\n\");\n\t\t\t\telse\n\t\t\t\t\tstrcat(msg3,\"(time of measurement can not be decreasing)\\n\");\n\t\t\t\tprintf(msg3);\n\t\t\t\tfprintf(fout,msg3);\n\t\t\t\tstrcpy(msg3,\"        --------------------------------------\\n\");\n\t\t\t\tstrcat(msg3,\"        |            |    day     |  minute  |\\n\");\n\t\t\t\tstrcat(msg3,\"        --------------------------------------\\n\");\n\t\t\t\tprintf(msg3);\n\t\t\t\tfprintf(fout,msg3);\n\t\t\t\tstrcpy(msg3,\"        | Previous   |  %5d     | %6d   |\\n\");\n\t\t\t\tprintf(msg3,day0,min0);\n\t\t\t\tfprintf(fout,msg3,day0,min0);\n\t\t\t\tstrcpy(msg3,\"        | Current    |  %5d     | %6d   |\\n\");\n\t\t\t\tstrcat(msg3,\"        --------------------------------------\\n\\n\");\n\t\t\t\tprintf(msg3,day1,min1);\n\t\t\t\tfprintf(fout,msg3,day1,min1);\n\t\t\t\tday0=-1;\n\t\t\t\tcontinue;\n\t\t\t\t}\n\t\t\telse\n\t\t\t\t{\n\t\t\t\tday0=day1;/* set cur. day and min. to prev.\t*/\n\t\t\t\tmin0=min1;\n\t\t\t\t}\n\t\t\t}\n\n\t\t/* wrong format of record line\t\t\t\t\t*/\n\t\tif(strncmp(file_out,buffer,strlen(file_out)) ||  strlen(buffer) != strlen(xformat) && strlen(buffer)<=80)\n\t\t\t{\n\t\t\tcount3++;\n\t\t\tif(count3==1)\n\t\t\t\t{\n\t\t\t\tprintf(\"%s\",msg1);\n\t\t\t\tfprintf(fout,\"%s\",msg1);\n\t\t\t\t}\n\t\t\tprintf(msg2,rec_num,count2);\n\t\t\tfprintf(fout,msg2,rec_num,count2);\n\n\t\t\t/* replace new line '\\n' by '|' to indicate the line end*/\n\t\t\tif((int)strlen(buffer)<=80)\n\t\t\t\tbuffer[strlen(buffer)-1]='|';\n\t\t\telse\n\t\t\t\tbuffer[strlen(buffer)-1]='\\0';\n\n\t\t\tstrcpy(msg3,\"        +Wrong format of the line...\\n\");\n\t\t\tstrcat(msg3,\"         In the following lines the expected format and the current\\n\");\n\t\t\tstrcat(msg3,\"         format of the line are printed.\\n\\n\");\n\t\t\tstrcat(msg3,\"         '#' stands for a digit (leading zeros are not allowed)\\n\");\n\t\t\tstrcat(msg3,\"         'X' stands for a character\\n\");\n\t\t\tstrcat(msg3,\"         '|' stands for the line end if it is present within first 80 characters\\n\");\n\t\t\tstrcat(msg3,\"             (additional characters or spaces are not allowed)\\n\");\n\t\t\tif(rec_num==8 && count1==4)\n\t\t\t\tif(band4==0)\n\t\t\t\t\tstrcat(msg3,\"3-BAND RADIATION INSTRUMENT?\\n\");\n\t\t\t\telse\n\t\t\t\t\tstrcat(msg3,\"4-BAND RADIATION INSTRUMENT?\\n\");\n\t\t\tstrcat(msg3,\"--------------------------------------------------------------------------------\\n\");\n\t\t\tstrcat(msg3,\"%s\\n\");\n\t\t\tstrcat(msg3,\"%s\\n\");\n\t\t\tstrcat(msg3,\"--------------------------------------------------------------------------------\\n\\n\");\n\t\t\tprintf(msg3,xformat,buffer);\n\t\t\tfprintf(fout,msg3,xformat,buffer);\n\t\t\tday0=-1;\n\n\t\t\tcontinue;\n\t\t\t}\n\t\t}\n\tif(count3==0)\n\t\t{\n\t\tstrcpy(msg2,\"*Check for line format.......... OK\\n\");\n\t\tprintf(msg2);\n\t\tfprintf(fout,msg2);\n\t\t}\n\telse\n\t\terr=1;\n\t}\nfclose(fin);\nfclose(fout);\n\nexit(err);\n}\n"

def write_c_source(output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    source_path = output_dir / "f_check_V3_4.c"
    source_path.write_text(C_SOURCE, encoding="utf-8", errors="replace")
    return source_path

## 3. Filename parsing and checker utilities

The BSRN archive filename convention is `sssmmyy.dat`, where `sss` is the station abbreviation, `mm` is month, and `yy` is two-digit year. The parser below is used only for reporting and input validation. The actual format check is done by the original checker.

In [4]:
@dataclass
class BSRNFileResult:
    input_file: str
    station: Optional[str]
    month: Optional[int]
    year_2digit: Optional[int]
    year_guess: Optional[int]
    filename_valid: bool
    report_file: Optional[str]
    return_code: Optional[int]
    line_length_ok: Optional[bool]
    illegal_chars_ok: Optional[bool]
    line_format_ok: Optional[bool]
    passed: bool
    message: str


def parse_bsrn_filename(path: str | Path):
    name = Path(path).name
    m = re.fullmatch(r"([A-Za-z0-9]{3})(\d{2})(\d{2})\.dat", name, flags=re.IGNORECASE)
    if not m:
        return None
    station, mm, yy = m.group(1).lower(), int(m.group(2)), int(m.group(3))
    year_guess = 1900 + yy if yy >= 92 else 2000 + yy
    return {"station": station, "month": mm, "year_2digit": yy, "year_guess": year_guess, "valid": 1 <= mm <= 12}


def resolve_inputs(inputs: Iterable[str | Path], input_dir: Optional[str | Path] = None) -> list[Path]:
    paths = [Path(item).expanduser() for item in (inputs or [])]
    if input_dir:
        paths.extend(sorted(Path(input_dir).expanduser().glob("*.dat")))
    out, seen = [], set()
    for p in paths:
        key = str(p.resolve()) if p.exists() else str(p.absolute())
        if key not in seen:
            out.append(p)
            seen.add(key)
    return out


def find_c_compiler() -> Optional[str]:
    for compiler in ("gcc", "cc", "clang"):
        path = shutil.which(compiler)
        if path:
            return path
    return None


def compile_checker(output_dir: str | Path = OUTPUT_DIR, force: bool = FORCE_RECOMPILE) -> Path:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    exe = output_dir / ("f_check.exe" if os.name == "nt" else "f_check")
    if exe.exists() and not force:
        return exe
    compiler = find_c_compiler()
    if compiler is None:
        raise RuntimeError("No C compiler found. Install gcc/cc/clang or run this notebook in an environment with a C compiler.")
    source_path = write_c_source(output_dir)
    last_proc = None
    for cmd in ([compiler, "-O2", "-std=c89", "-w", "-o", str(exe), str(source_path)],
                [compiler, "-O2", "-w", "-o", str(exe), str(source_path)]):
        last_proc = subprocess.run(cmd, text=True, capture_output=True)
        if last_proc.returncode == 0:
            return exe
    raise RuntimeError(f"Compilation failed.\nSTDOUT:\n{last_proc.stdout}\nSTDERR:\n{last_proc.stderr}")


def parse_report(rep_text: str):
    return {
        "line_length_ok": "*Check for line length.......... OK" in rep_text,
        "illegal_chars_ok": "*Check for illegal characters... OK" in rep_text,
        "line_format_ok": "*Check for line format.......... OK" in rep_text,
    }



def expected_rep_path(dat_file: Path) -> Path:
    """The legacy checker writes <input>.rep next to <input>.dat."""
    return dat_file.with_suffix(".rep")


def output_rep_path(dat_file: Path, output_dir: str | Path) -> Path:
    """Canonical report location used by this notebook."""
    return Path(output_dir) / f"{dat_file.stem}.rep"


def check_file(
    dat_file: str | Path,
    output_dir: str | Path = OUTPUT_DIR,
    exe: Optional[Path] = None,
    remove_input_side_report: bool = REMOVE_INPUT_SIDE_REPORTS,
) -> BSRNFileResult:
    dat_file = Path(dat_file).expanduser()
    meta = parse_bsrn_filename(dat_file)
    station = meta["station"] if meta else None
    month = meta["month"] if meta else None
    yy = meta["year_2digit"] if meta else None
    year_guess = meta["year_guess"] if meta else None
    filename_valid = bool(meta and meta["valid"])

    out_dir = Path(output_dir).expanduser()
    out_dir.mkdir(parents=True, exist_ok=True)
    rep_dest = output_rep_path(dat_file, out_dir)

    if not dat_file.exists():
        rep_text = f"Input file not found: {dat_file}\n"
        rep_dest.write_text(rep_text, encoding="utf-8", errors="replace")
        return BSRNFileResult(
            str(dat_file), station, month, yy, year_guess, filename_valid,
            str(rep_dest), None, None, None, None, False,
            "Input file not found; check INPUTS/INPUT_DIR path"
        )

    exe = exe or compile_checker(out_dir)

    # Run the original checker. It writes the .rep file next to the .dat file.
    proc = subprocess.run([str(exe), str(dat_file)], text=True, capture_output=True)
    rep_src = expected_rep_path(dat_file)

    if rep_src.exists():
        rep_text = rep_src.read_text(errors="replace")
        rep_dest.write_text(rep_text, encoding="utf-8", errors="replace")

        # Keep reports only in OUTPUT_DIR. If input and output folders are the same,
        # this is a no-op and the report is not deleted.
        if remove_input_side_report:
            try:
                if rep_src.resolve() != rep_dest.resolve():
                    rep_src.unlink()
            except OSError as exc:
                print(f"Warning: could not remove temporary input-side report {rep_src}: {exc}")
    else:
        rep_text = proc.stdout + "\n" + proc.stderr
        rep_dest.write_text(rep_text, encoding="utf-8", errors="replace")

    flags = parse_report(rep_text)
    passed = proc.returncode == 0 and all(flags.values())
    message = "Passed original f_check V3.4 checks" if passed else "Failed one or more original f_check V3.4 checks; inspect report"
    if not filename_valid:
        message += "; filename does not match sssmmyy.dat with valid month"

    return BSRNFileResult(
        input_file=str(dat_file), station=station, month=month, year_2digit=yy, year_guess=year_guess,
        filename_valid=filename_valid, report_file=str(rep_dest), return_code=proc.returncode,
        line_length_ok=flags["line_length_ok"], illegal_chars_ok=flags["illegal_chars_ok"],
        line_format_ok=flags["line_format_ok"], passed=passed, message=message
    )


def write_combined_report(
    results: Iterable[BSRNFileResult],
    output_dir: str | Path = OUTPUT_DIR,
    combined_report_file: Optional[str] = COMBINED_REPORT_FILE,
) -> Optional[Path]:
    """Append individual .rep contents into one combined .rep file.

    The individual report lines are appended in the same order as `results`.
    A short file separator is inserted before each report so the origin remains clear.
    """
    if not combined_report_file:
        return None

    out_dir = Path(output_dir).expanduser()
    out_dir.mkdir(parents=True, exist_ok=True)
    combined_path = out_dir / combined_report_file

    chunks = []
    for result in results:
        chunks.append(f"\n{'=' * 80}\n")
        chunks.append(f"Input file: {result.input_file}\n")
        chunks.append(f"Report file: {result.report_file}\n")
        chunks.append(f"{'=' * 80}\n")

        if result.report_file and Path(result.report_file).exists():
            text = Path(result.report_file).read_text(errors="replace")
            chunks.append(text)
            if text and not text.endswith("\n"):
                chunks.append("\n")
        else:
            chunks.append(result.message + "\n")

    combined_path.write_text("".join(chunks).lstrip("\n"), encoding="utf-8", errors="replace")
    return combined_path


def check_many(
    inputs: Iterable[str | Path],
    input_dir: Optional[str | Path] = None,
    output_dir: str | Path = OUTPUT_DIR,
    remove_input_side_reports: bool = REMOVE_INPUT_SIDE_REPORTS,
    combined_report_file: Optional[str] = COMBINED_REPORT_FILE,
):
    files = resolve_inputs(inputs, input_dir)
    exe = compile_checker(output_dir)
    results = [
        check_file(
            p,
            output_dir=output_dir,
            exe=exe,
            remove_input_side_report=remove_input_side_reports,
        )
        for p in files
    ]
    combined_path = write_combined_report(results, output_dir, combined_report_file)
    if combined_path:
        print(f"Combined report written to: {combined_path}")
    return results


## 4. Run checks

This cell runs the checker for all configured inputs. It works the same for every station abbreviation, month, and year, as long as the file is a BSRN station-to-archive `.dat` file.

In [5]:
results = check_many(INPUTS, input_dir=INPUT_DIR, output_dir=OUTPUT_DIR)

if pd is not None:
    summary = pd.DataFrame([asdict(r) for r in results])
    display(summary)
else:
    for r in results:
        print(asdict(r))

Combined report written to: bsrn_check_output/combined_checks.rep


,input_file,station,month,year_2digit,year_guess,filename_valid,report_file,return_code,line_length_ok,illegal_chars_ok,line_format_ok,passed,message
0,bsrn_check_input/dra0125.dat,dra,1,25,2025,True,bsrn_check_output/dra0125.rep,0,True,True,True,True,Passed original f_check V3.4 checks
1,bsrn_check_input/dra0425.dat,dra,4,25,2025,True,bsrn_check_output/dra0425.rep,0,True,True,True,True,Passed original f_check V3.4 checks
2,bsrn_check_input/dra0525.dat,dra,5,25,2025,True,bsrn_check_output/dra0525.rep,0,True,True,True,True,Passed original f_check V3.4 checks
3,bsrn_check_input/dra0625.dat,dra,6,25,2025,True,bsrn_check_output/dra0625.rep,0,True,True,True,True,Passed original f_check V3.4 checks
4,bsrn_check_input/dra0725.dat,dra,7,25,2025,True,bsrn_check_output/dra0725.rep,0,True,True,True,True,Passed original f_check V3.4 checks
5,bsrn_check_input/dra0825.dat,dra,8,25,2025,True,bsrn_check_output/dra0825.rep,0,True,True,True,True,Passed original f_check V3.4 checks
6,bsrn_check_input/dra0925.dat,dra,9,25,2025,True,bsrn_check_output/dra0925.rep,0,True,True,True,True,Passed original f_check V3.4 checks
7,bsrn_check_input/dra1025.dat,dra,10,25,2025,True,bsrn_check_output/dra1025.rep,0,True,True,True,True,Passed original f_check V3.4 checks


## 5. Inspect reports

The original `.rep` report is the authoritative output. This helper prints the report for one selected result.

In [6]:
def show_report(result_index: int = 0, max_chars: Optional[int] = None):
    """Print one checker report safely.

    If a check failed before a .rep file could be generated, this function
    prints the stored result message instead of raising a TypeError.
    """
    if not results:
        print("No results available. Run the check cell first.")
        return

    if result_index < 0 or result_index >= len(results):
        print(f"Result index {result_index} is out of range. Available: 0..{len(results)-1}.")
        return

    result = results[result_index]

    if not result.report_file:
        print(result.message)
        print(f"Input file: {result.input_file}")
        return

    path = Path(result.report_file)
    if not path.exists():
        print(f"Report file does not exist: {path}")
        print(result.message)
        print(f"Input file: {result.input_file}")
        return

    text = path.read_text(errors="replace")
    print(text if max_chars is None else text[:max_chars])

show_report(0)



**********
File name: dra0125.dat
**********
*Check for line length.......... OK
*Check for illegal characters... OK
*Check for line format.......... OK



## 6. Batch mode examples

Use one of these patterns when checking deliveries from multiple stations or months.

In [7]:
# Example A: check selected files
# INPUTS = ["dra0125.dat", "abc0225.dat", "xyz1224.dat"]
# results = check_many(INPUTS, output_dir=OUTPUT_DIR)

# Example B: check all .dat files in a folder and keep reports only in OUTPUT_DIR
# INPUTS = []
# INPUT_DIR = "bsrn_check_input"
# OUTPUT_DIR = "bsrn_check_output"
# COMBINED_REPORT_FILE = "combined_checks.rep"
# results = check_many(INPUTS, input_dir=INPUT_DIR, output_dir=OUTPUT_DIR)

# Example C: disable combined output
# results = check_many(INPUTS, input_dir=INPUT_DIR, output_dir=OUTPUT_DIR, combined_report_file=None)

# Example D: fail the notebook if any file fails, useful for automated QA
# assert all(r.passed for r in results), "At least one BSRN file failed the original f_check V3.4 checks"


## 7. Verification notes

This notebook handles different stations, months, and years by design because the checker is called on the input path, not on a hard-coded filename. The filename parser is used only to summarize station/month/year and to flag invalid archive-style filenames. The original C checker validates the file contents, including logical record `0001`, the date fields, line lengths, illegal characters, and record-specific formats.

Known operational dependency: a C compiler must be available in the Jupyter environment. If a no-compiler deployment is required later, the next step is to distribute a precompiled binary per platform or finish a pure-Python port verified against this notebook as oracle.